# EEG / Frontal Alpha Asymmetry pipeline — pricing-related consumer decisions

Reproducible analysis notebook for the manuscript. It loads the de-identified single-trial EEG (`data/csv/`), the precomputed ICA solutions (`data/ICA/`), and the behavioral ratings (`data/raw/<subject>/`), and reproduces: behavioral descriptives (Objective 1); the GEE / cluster-robust OLS models in Tables 1–3 and the 3-way interaction (Objective 2); the single-trial classification across momentum thresholds (Objective 3); and Figure 3 / S1 Fig.

Subjects/sessions are identified only by anonymous codes `subjNN_sessM`.

In [1]:
# Cell 1 — Imports & configuration
import os, re, warnings
import numpy as np
import pandas as pd
import mne
warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

DATA = "./data"
csv_dir   = f"{DATA}/csv"
ica_dir   = f"{DATA}/ICA"
rating_dir= f"{DATA}/raw"
os.makedirs("fig", exist_ok=True)

sfreq = 512.0
# CSV has Fp1..AF4 + Oz; Oz is dropped -> 15-ch 10-20 montage (matches paper)
ch_names_eeg = ["Fp1","Fp2","F7","F8","F3","Fz","F4","C3","Cz","C4","P3","Pz","P4","AF3","AF4","Oz"]
event_dict = {"show_fixation1-1":1,"show_product1":2,"show_product_scale1":3,
              "show_fixation1-2":4,"show_price1":5,"show_price_scale1":6}
PRICE_ID, PRODUCT_ID = 5, 2

# Rating/raw folders are named subjNN_sessM (de-identified); derived from CSV id+session in Cell 5.

# ICA components to remove per recording, selected by visual inspection (index -> [components]).
# Corrected after visual re-review: idx 4 (subj03_sess1) [1]->[0], idx 40 (subj21_sess1) [0]->[4].
exclude_dict = {0:[0,1],1:[0,1],2:[0],3:[0],4:[0],5:[0],6:[0],7:[0],8:[0],9:[0],10:[0],11:[1],
 12:[0],13:[0],14:[0],15:[0],16:[0],17:[0],18:[0],19:[0],20:[0,1],21:[0],22:[0,1],23:[0],
 24:[0,1],25:[0],26:[0],27:[0],28:[0,1],29:[2],30:[0],31:[0],32:[0],33:[0],34:[0],35:[0],
 36:[0],37:[0],38:[0],39:[0],40:[4],41:[0]}

## Data preparation — Raw LabRecorder XDF → CSV

Converts the raw `.xdf` recordings under `data/raw/<id>/<id>.xdf` into the per-recording CSVs (`data/csv/<id>.csv`) consumed below, so the pipeline is reproducible from raw. **Idempotent** — recordings whose CSV already exists are skipped, and `pyxdf` is imported only when a conversion is actually needed.

In [2]:
# Cell 1b — Raw LabRecorder XDF -> per-recording CSV (reproducibility from raw)
# Each data/raw/<id>/<id>.xdf holds a "designer" EEG stream (16 ch, microvolts) and a
# "PsychoPyMarkers" event stream. We write data/csv/<id>.csv = 16 EEG channels + an LSL
# "timestamp" column + a "trigger" column, placing each marker on its NEAREST EEG sample.
# Idempotent: recordings whose CSV already exists are skipped (delete a CSV to regenerate it).

def xdf_to_csv(xdf_path, out_csv):
    from pyxdf import load_xdf                                  # imported lazily (only when converting)
    streams, _ = load_xdf(xdf_path)
    eeg = next(s for s in streams if s["info"]["name"][0] == "designer")
    mrk = next(s for s in streams if s["info"]["name"][0] == "PsychoPyMarkers")
    ets = np.asarray(eeg["time_stamps"]); mts = np.asarray(mrk["time_stamps"])
    df = pd.DataFrame(eeg["time_series"], columns=ch_names_eeg)  # raw microvolts (DC removed in Cell 2)
    df["timestamp"] = ets; df["trigger"] = 0
    nn = np.clip(np.searchsorted(ets, mts), 1, len(ets) - 1)     # nearest EEG sample to each marker
    nn = np.where(np.abs(ets[nn] - mts) < np.abs(ets[nn - 1] - mts), nn, nn - 1)
    df.iloc[nn, df.columns.get_loc("trigger")] = [int(m[0]) for m in mrk["time_series"]]
    df.to_csv(out_csv, index=False)

os.makedirs(csv_dir, exist_ok=True)
_new = 0
for _stem in sorted(d for d in os.listdir(rating_dir) if re.fullmatch(r"subj\d+_sess\d+", d)):
    _xdf = f"{rating_dir}/{_stem}/{_stem}.xdf"; _csv = f"{csv_dir}/{_stem}.csv"
    if os.path.exists(_xdf) and not os.path.exists(_csv):
        xdf_to_csv(_xdf, _csv); _new += 1

file_list = sorted(f for f in os.listdir(csv_dir) if f.endswith('.csv'))
print(f"XDF->CSV: {_new} new CSV(s) generated | {len(file_list)} CSV recordings ready")

XDF->CSV: 0 new CSV(s) generated | 42 CSV recordings ready


In [3]:
# Cell 2 — Load CSV -> Raw, filter 0.3-45 Hz (+notch 50), find events
raw_list, events_list, subject_ids_list, session_ids_list = [], [], [], []
for fn in file_list:
    m = re.search(r"subj(\d+)_sess(\d+)", fn)
    if not m:
        continue
    s_id, sess = int(m.group(1)), int(m.group(2))
    df = pd.read_csv(os.path.join(csv_dir, fn))

    eeg = df[ch_names_eeg].values.T
    eeg = eeg - eeg.mean(axis=1, keepdims=True)   # remove per-channel DC offset
    eeg = eeg * 1e-6                              # uV -> V
    trig = np.round(df['trigger'].values).astype(int).reshape(1, -1)

    info = mne.create_info(ch_names_eeg + ['trigger'], sfreq, ['eeg']*16 + ['stim'])
    raw = mne.io.RawArray(np.vstack([eeg, trig]), info)
    raw.drop_channels(['Oz'])
    raw.set_montage(mne.channels.make_standard_montage("standard_1020"))

    events = mne.find_events(raw, stim_channel="trigger", min_duration=0, verbose=False)
    raw.filter(0.3, 45.0, verbose=False)
    raw.notch_filter(50, verbose=False)          # (redundant after 45 Hz LP; kept for parity)
    raw.drop_channels(['trigger'])

    raw_list.append(raw); events_list.append(events)
    subject_ids_list.append(s_id); session_ids_list.append(sess)
print(f"Loaded {len(raw_list)} recordings")

Loaded 42 recordings


In [4]:
# Cell 3 — ICA-copy segment QC (amplitude reject + 20% channel rule; documents Methods)
# In this dataset no channel met the 20% bad-epoch threshold, so none were excluded.
ica_epochs_list = []
print(f"{'Subj':<5}{'Sess':<5}{'Init':<6}{'AfterAmp':<9}{'Excluded ch'}")
for idx, raw in enumerate(raw_list):
    raw_ica = raw.copy().filter(1.0, 45.0, verbose=False)
    ep = mne.make_fixed_length_epochs(raw_ica, duration=1.0, preload=True, verbose=False)
    total = len(ep)
    ep.drop_bad(reject=dict(eeg=1000e-6), verbose=False)   # amplitude rejection (1000 uV peak-to-peak)
    after_amp = len(ep)
    # 20% channel rule (safeguard; none excluded here)
    excl = [c for c in ep.ch_names if sum(1 for lg in ep.drop_log if c in lg)/total > 0.20]
    ica_epochs_list.append(ep)
    print(f"P{subject_ids_list[idx]:<4}S{session_ids_list[idx]:<4}{total:<6}{after_amp:<9}{excl if excl else '-'}")

Subj Sess Init  AfterAmp Excluded ch


P1   S1   537   536      -


P1   S2   497   497      -


P2   S1   518   514      -


P2   S2   486   482      -
P3   S1   452   442      -


...

P17  S1   572   541      -
P17  S2   572   537      -


P18  S1   563   563      -
P18  S2   487   467      -


P19  S1   641   641      -
P19  S2   575   574      -


P20  S1   611   606      -
P20  S2   521   517      -


P21  S1   510   510      -
P21  S2   502   502      -


In [5]:
# Cell 4 — Load precomputed ICA (extended infomax, 15 comp) from disk
ica_list = []
for idx in range(len(raw_list)):
    fp = os.path.join(ica_dir, f"subj{subject_ids_list[idx]:02d}_sess{session_ids_list[idx]}-ica.fif")
    ica_list.append(mne.preprocessing.read_ica(fp, verbose=False))
print(f"Loaded {sum(x is not None for x in ica_list)} ICA solutions")

Loaded 42 ICA solutions


In [6]:
# Cell 5 — Apply ICA -> avg ref -> price epochs (-1..3), QC from task window (0..3) @150uV
# Includes a ratings.csv <-> image_order.csv ordering cross-check (imgChk).
EPOCH_TMIN, EPOCH_TMAX = -1.0, 3.0
QC_TMIN, QC_TMAX = 0.0, 3.0
reject_uV = 150e-6

def trim_to_n(ev, n):
    return ev[:n] if (ev is not None and len(ev) > n) else ev

price_epochs_list = []
qc_flow = []  # A3: participant/trial-flow accounting
print(f"{'Idx':<4}{'Subj':<6}{'Sess':<5}{'Keep':<6}{'imgChk':<8}Status")
for i, raw_obj in enumerate(raw_list):
    sid, sess = subject_ids_list[i], session_ids_list[i]
    p_name = f"subj{sid:02d}_sess{sess}"
    r_df = pd.read_csv(os.path.join(rating_dir, p_name, 'rating_scale', 'ratings.csv'))
    n_trials = len(r_df)

    # cross-check: ratings order == presentation order (image_order.csv)
    img_chk = "n/a"
    io_path = os.path.join(rating_dir, p_name, 'image_order.csv')
    if os.path.exists(io_path):
        io = pd.read_csv(io_path)
        if 'image_id' in io.columns and 'image_id' in r_df.columns:
            img_chk = "OK" if list(io['image_id'])[:n_trials] == list(r_df['image_id']) else "MISMATCH!"

    meta = []
    for t in range(n_trials):
        pr = float(r_df.loc[t,'rating_product']); pp = float(r_df.loc[t,'rating_price'])
        diff = pp - pr
        if pr == 1 and pp == 1: diff = -1
        elif pr == 9 and pp == 9: diff = 1
        meta.append({"trial_index":t,"subject_id":sid,"session":sess,
                     "rating_product":pr,"rating_price":pp,"momentum":float(abs(diff)),
                     "decision":"Positive" if diff>0 else "Negative","diff_score":float(diff),
                     "image_id":r_df.loc[t,"image_id"] if "image_id" in r_df.columns else t})
    metadata_df = pd.DataFrame(meta).set_index("trial_index")

    temp = ica_list[i].apply(raw_obj.copy(), exclude=exclude_dict.get(i, []), verbose=False)
    if temp.info.get("bads"):
        temp.interpolate_bads(reset_bads=True, verbose=False)
    temp.set_eeg_reference("average", projection=False, verbose=False)

    price = trim_to_n(events_list[i][events_list[i][:,2]==PRICE_ID], n_trials)  # extra marker is trailing
    mp = metadata_df.iloc[:len(price)].copy()
    ep = mne.Epochs(temp, price, event_id={"show_price1":PRICE_ID},
                    tmin=EPOCH_TMIN, tmax=EPOCH_TMAX, baseline=None,
                    metadata=mp, preload=True, verbose=False)
    made = len(ep)
    task = ep.copy().crop(QC_TMIN, QC_TMAX).get_data()
    ptp = task.max(-1) - task.min(-1)
    bad = np.where(np.any(ptp > reject_uV, axis=1))[0]
    if len(bad) > 0:
        ep.drop(bad, verbose=False)
    price_epochs_list.append(ep)
    qc_flow.append(dict(subj=sid, sess=sess, n_trials=n_trials, made=made,
                        dropped_150uV=int(len(bad)), kept=len(ep),
                        n_ica_removed=len(exclude_dict.get(i, []))))
    print(f"{i:<4}P{sid:<5}S{sess:<4}{len(ep):<6}{img_chk:<8}OK")

Idx Subj  Sess Keep  imgChk  Status
0   P1    S1   25    OK      OK
1   P1    S2   18    OK      OK


2   P2    S1   30    OK      OK
3   P2    S2   29    OK      OK
4   P3    S1   22    OK      OK


5   P3    S2   30    OK      OK
6   P4    S1   30    OK      OK
7   P4    S2   30    OK      OK


8   P5    S1   17    OK      OK
9   P5    S2   19    OK      OK
10  P6    S1   30    OK      OK


11  P6    S2   24    OK      OK
12  P7    S1   28    OK      OK
13  P7    S2   28    OK      OK


...

26  P14   S1   27    OK      OK
27  P14   S2   30    OK      OK
28  P15   S1   21    OK      OK
29  P15   S2   13    OK      OK


30  P16   S1   24    OK      OK
31  P16   S2   28    OK      OK
32  P17   S1   26    OK      OK


33  P17   S2   18    OK      OK
34  P18   S1   21    OK      OK
35  P18   S2   19    OK      OK


36  P19   S1   30    OK      OK
37  P19   S2   29    OK      OK
38  P20   S1   20    OK      OK


39  P20   S2   29    OK      OK
40  P21   S1   0     OK      OK
41  P21   S2   1     OK      OK


In [7]:
# Cell 6 — FAA_post (0..3) and FAA_pre (-1..0), BOTH n_fft=512, F3/F4 by NAME
def rel_alpha_log(psd_a, psd_t, idx):
    return np.log(psd_a[:, idx] / psd_t[:, idx])

def faa_window(ep, tmin, tmax, n_fft, left="F3", right="F4"):
    e = ep.copy().crop(tmin, tmax)
    li, ri = e.ch_names.index(left), e.ch_names.index(right)
    a = e.compute_psd(method="welch", fmin=8,  fmax=13, n_fft=n_fft, verbose=False).get_data().mean(-1)
    t = e.compute_psd(method="welch", fmin=1,  fmax=40, n_fft=n_fft, verbose=False).get_data().mean(-1)
    return rel_alpha_log(a, t, ri) - rel_alpha_log(a, t, li)  # relative-alpha FAA for an arbitrary L/R pair

def abs_faa_window(ep, tmin, tmax, n_fft):
    # Absolute-alpha FAA = ln(absAlpha_F4) - ln(absAlpha_F3) (A4 sensitivity: reference-free of total-power denominator)
    e = ep.copy().crop(tmin, tmax)
    f3, f4 = e.ch_names.index("F3"), e.ch_names.index("F4")
    a = e.compute_psd(method="welch", fmin=8, fmax=13, n_fft=n_fft, verbose=False).get_data().mean(-1)
    return np.log(a[:, f4]) - np.log(a[:, f3])

# Posterior alpha power (absolute log, 8-13 Hz) over centro-parietal sensors:
# an objective cortical-arousal index (higher = lower arousal / drowsier). Same
# Welch settings/window as FAA_post for comparability.
POSTERIOR_ROI = ["P3", "Pz", "P4", "C3", "Cz", "C4"]
def post_alpha_window(ep, tmin, tmax, n_fft):
    e = ep.copy().crop(tmin, tmax)
    idx = [e.ch_names.index(c) for c in POSTERIOR_ROI]
    a = e.compute_psd(method="welch", fmin=8, fmax=13, n_fft=n_fft, verbose=False).get_data().mean(-1)
    return np.log(a[:, idx]).mean(1)

records = []
for ep in price_epochs_list:
    if ep is None or len(ep) == 0:
        continue
    faa_post = faa_window(ep, 0.0, 3.0, 512)
    faa_pre  = faa_window(ep, -1.0, 0.0, 512)   # n_fft=512 (same as FAA_post) for comparability across windows
    faa_abs  = abs_faa_window(ep, 0.0, 3.0, 512)  # A4 sensitivity
    faa_f7f8 = faa_window(ep, 0.0, 3.0, 512, left="F7", right="F8")   # B-S4 electrode specificity
    faa_af   = faa_window(ep, 0.0, 3.0, 512, left="AF3", right="AF4")  # B-S4 electrode specificity
    post_alpha = post_alpha_window(ep, 0.0, 3.0, 512)
    md = ep.metadata.reset_index(drop=True)
    for j in range(len(ep)):
        m = md.iloc[j]
        records.append({
            "Subject_ID": f"S{int(m['subject_id']):02d}",
            "Session": int(m['session']),
            "Valence": "Positive" if m['diff_score'] > 0 else "Negative",
            "Momentum": float(m['momentum']),
            "RatingProduct": float(m['rating_product']),
            "RatingPrice": float(m['rating_price']),
            "ImageID": m['image_id'],
            "FAA": float(faa_post[j]),
            "FAA_pre": float(faa_pre[j]),
            "FAA_abs": float(faa_abs[j]),
            "FAA_F7F8": float(faa_f7f8[j]),
            "FAA_AF3AF4": float(faa_af[j]),
            "PosteriorAlpha": float(post_alpha[j]),
        })
df_all = pd.DataFrame(records)
df_all["dFAA"] = df_all["FAA"] - df_all["FAA_pre"]

# Filter: >=3 Positive AND >=3 Negative in BOTH sessions
keep = []
for sub in df_all["Subject_ID"].unique():
    s = df_all[df_all.Subject_ID == sub]
    ok = s.Session.nunique() == 2
    for ss in (1, 2):
        d = s[s.Session == ss]
        if (d.Valence == "Positive").sum() < 3 or (d.Valence == "Negative").sum() < 3:
            ok = False
    if ok:
        keep.append(sub)
df_f = df_all[df_all.Subject_ID.isin(keep)].copy()
for col, cats in [("Valence", ["Positive","Negative"]), ("Session", [1,2])]:
    df_f[col] = pd.Categorical(df_f[col], categories=cats)

print("All trials:", len(df_all), "| Filtered:", len(df_f), "| Subjects:", df_f.Subject_ID.nunique())
print("Valence:", df_f.Valence.value_counts().to_dict())
print("Session:", df_f.Session.value_counts().to_dict())

# Create df_primary by excluding boundary trials
bmask = ((df_f.RatingProduct == 9) & (df_f.RatingPrice == 9)) | ((df_f.RatingProduct == 1) & (df_f.RatingPrice == 1))
df_primary = df_f[~bmask].copy()
print(f"df_primary (no boundary): {len(df_primary)}")


All trials: 959 | Filtered: 937 | Subjects: 19
Valence: {'Negative': 479, 'Positive': 458}
Session: {1: 474, 2: 463}
df_primary (no boundary): 820


In [8]:
# Cell 6b — Participant/trial-flow accounting (Methods exclusion flow) [A3a/A3b]
fl = pd.DataFrame(qc_flow)
n_part = fl.subj.nunique()
made = int(fl.made.sum()); dropped = int(fl.dropped_150uV.sum()); kept = int(fl.kept.sum())
print("PARTICIPANT / TRIAL FLOW")
print(f"  Possible trials: {n_part} participants x 60 trials = {n_part*60}")
print(f"  Epochs created (price-locked):            {made}")
print(f"  Rejected by 150 uV peak-to-peak (0-3 s):  {dropped}  ({100*dropped/made:.1f}%)")
print(f"  Retained after artifact rejection:        {kept}   -> df_all rows: {len(df_all)}")
dropped_subj = sorted(set(df_all.Subject_ID.unique()) - set(keep))
print(f"  Screening rule: >=3 Positive AND >=3 Negative trials in BOTH sessions")
print(f"  Excluded by screening: {dropped_subj} ({len(dropped_subj)} participants, {len(df_all)-len(df_f)} trials)")
print(f"  -> FINAL analysis set: {len(df_f)} trials, {df_f.Subject_ID.nunique()} participants")
print(f"  ICA components removed per recording: {fl.n_ica_removed.value_counts().to_dict()} (total {int(fl.n_ica_removed.sum())})")
b99 = int(((df_f.RatingProduct==9)&(df_f.RatingPrice==9)).sum())
b11 = int(((df_f.RatingProduct==1)&(df_f.RatingPrice==1)).sum())
print(f"  Scale-boundary trials in final set: 9->9 (Positive)={b99}, 1->1 (Negative)={b11}")


PARTICIPANT / TRIAL FLOW
  Possible trials: 21 participants x 60 trials = 1260
  Epochs created (price-locked):            1260
  Rejected by 150 uV peak-to-peak (0-3 s):  301  (23.9%)
  Retained after artifact rejection:        959   -> df_all rows: 959
  Screening rule: >=3 Positive AND >=3 Negative trials in BOTH sessions
  Excluded by screening: ['S08', 'S21'] (2 participants, 22 trials)
  -> FINAL analysis set: 937 trials, 19 participants
  ICA components removed per recording: {1: 36, 2: 6} (total 48)
  Scale-boundary trials in final set: 9->9 (Positive)=13, 1->1 (Negative)=104


In [9]:
# Cell 6c — Artifact rejection bias check
import scipy.stats as stats
import statsmodels.api as sm

print("=== Artifact Rejection Bias Check ===")
reject_rows = []
for i, raw_obj in enumerate(raw_list):
    sid, sess = subject_ids_list[i], session_ids_list[i]
    p_name = f"subj{sid:02d}_sess{sess}"
    r_df = pd.read_csv(os.path.join(rating_dir, p_name, 'rating_scale', 'ratings.csv'))
    
    temp = ica_list[i].apply(raw_obj.copy(), exclude=exclude_dict.get(i, []), verbose=False)
    temp.set_eeg_reference("average", projection=False, verbose=False)
    price = trim_to_n(events_list[i][events_list[i][:,2]==PRICE_ID], len(r_df))
    
    ep = mne.Epochs(temp, price, event_id={"show_price1":PRICE_ID},
                    tmin=0.0, tmax=3.0, baseline=None, preload=True, verbose=False)
    task = ep.get_data(copy=False)
    ptp = task.max(-1) - task.min(-1)
    bad = np.any(ptp > reject_uV, axis=1)
    
    for t in range(len(ep)):
        pr = float(r_df.loc[t,'rating_product']); pp = float(r_df.loc[t,'rating_price'])
        diff = pp - pr
        mom = float(abs(diff))
        reject_rows.append({
            "subject": sid, "session": sess, 
            "valence": "Positive" if diff > 0 else "Negative",
            "momentum": mom, "rejected": bad[t]
        })

rej_df = pd.DataFrame(reject_rows)
print(f"Total trials checked: {len(rej_df)}, Rejected: {rej_df.rejected.sum()}")

ct_sess = pd.crosstab(rej_df.session, rej_df.rejected)
chi2_s, p_s, _, _ = stats.chi2_contingency(ct_sess)
print(f"Rejection by Session: p={p_s:.4f}")

ct_val = pd.crosstab(rej_df.valence, rej_df.rejected)
chi2_v, p_v, _, _ = stats.chi2_contingency(ct_val)
print(f"Rejection by Valence: p={p_v:.4f}")

mom_rej = rej_df[rej_df.rejected == True].momentum
mom_keep = rej_df[rej_df.rejected == False].momentum
t_m, p_m = stats.ttest_ind(mom_rej, mom_keep, equal_var=False)
print(f"Rejection by Momentum: p={p_m:.4f}")


=== Artifact Rejection Bias Check ===


Total trials checked: 1260, Rejected: 301
Rejection by Session: p=1.0000
Rejection by Valence: p=0.9420
Rejection by Momentum: p=0.0749


## Objective 1 — Behavioral shifts & descriptives

In [10]:
# Cell 7 — Objective 1: behavioral descriptives
import matplotlib.pyplot as plt, seaborn as sns
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(data=df_f, x="Valence", hue="Valence",
              palette={"Positive":"#2ecc71","Negative":"#e74c3c"}, ax=ax[0], legend=False)
ax[0].set_title("Proportion of Valence Shifts"); ax[0].set_ylabel("Trials")
sns.histplot(data=df_f, x="Momentum", discrete=True, color="#3498db", ax=ax[1])
ax[1].set_title("Decision Momentum |Δ|"); ax[1].set_xlabel("Momentum")
plt.tight_layout(); plt.savefig("fig/Fig_behavioral.png", dpi=150); plt.show()

print("Valence counts:\n", df_f.Valence.value_counts().to_string())
print(f"\nMean Momentum = {df_f.Momentum.mean():.4f}  SD = {df_f.Momentum.std():.4f}")

Valence counts:
 Valence
Negative    479
Positive    458

Mean Momentum = 1.8719  SD = 1.3328


## Objective 2 — Robust models (GEE primary, OLS cluster-robust sensitivity)

In [11]:
# Cell 8 — Shared robust-model helpers (GEE + OLS cluster-robust)
import statsmodels.formula.api as smf
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.cov_struct import Exchangeable

def fit_gee(formula, data):
    return GEE.from_formula(formula, groups="Subject_ID", data=data,
                            cov_struct=Exchangeable(), family=Gaussian()).fit()

def fit_ols_cluster(formula, data):
    return smf.ols(formula, data=data).fit(cov_type="cluster",
                                           cov_kwds={"groups": data["Subject_ID"]})

def tidy(res, label):
    ci = res.conf_int()
    out = pd.DataFrame({
        "Estimator": label,
        "Term": res.params.index,
        "Coef": res.params.values.round(4),
        "SE": res.bse.values.round(4),
        "p": res.pvalues.values.round(4),
        "CI2.5": ci[0].values.round(4),
        "CI97.5": ci[1].values.round(4),
    }).reset_index(drop=True)
    return out

def report(formula, data, title):
    print("="*92); print(title); print("  formula:", formula); print("-"*92)
    tbl = pd.concat([tidy(fit_gee(formula, data), "GEE"),
                     tidy(fit_ols_cluster(formula, data), "OLS-cluster")], ignore_index=True)
    with pd.option_context("display.float_format", "{:.4f}".format):
        print(tbl.to_string(index=False))
    return tbl

In [12]:
# Cell 9 — TABLE 1: Pooled CDM (primary momentum set, N=820)
_ = report("FAA ~ Valence*Momentum + RatingProduct + Session", df_primary,
           "TABLE 1 — CDM (pooled): FAA ~ Valence*Momentum + RatingProduct + Session")
print()
# The planned Dual-Rating Extended Model (DRE; + RatingPrice) does NOT converge on the primary
# momentum set (N=820): RatingPrice SE explodes and other SEs become NaN, reflecting shared
# variance among RatingProduct, RatingPrice, Valence, and Momentum. The paper reports this as a
# one-line non-convergence note (DRE not used for inference; CDM retained as the primary spec).
_dre820 = fit_gee("FAA ~ Valence*Momentum + RatingProduct + RatingPrice + Session", df_primary)
_se = np.asarray(_dre820.bse, float)
print("DRE non-convergence check on primary set (N=820):")
print(f"  max|SE| = {np.nanmax(np.abs(_se)):.1f}  |  any NaN SE: {bool(np.isnan(_se).any())}  "
      f"|  RatingPrice SE = {_dre820.bse.get('RatingPrice', float('nan')):.1f}")


TABLE 1 — CDM (pooled): FAA ~ Valence*Momentum + RatingProduct + Session
  formula: FAA ~ Valence*Momentum + RatingProduct + Session
--------------------------------------------------------------------------------------------
  Estimator                         Term    Coef     SE      p   CI2.5  CI97.5
        GEE                    Intercept -0.1013 0.0656 0.1225 -0.2299  0.0273
        GEE          Valence[T.Negative]  0.1352 0.0670 0.0435  0.0039  0.2665
        GEE                 Session[T.2] -0.0815 0.0470 0.0832 -0.1737  0.0107
        GEE                     Momentum  0.0511 0.0263 0.0524 -0.0005  0.1026
        GEE Valence[T.Negative]:Momentum -0.0510 0.0304 0.0932 -0.1106  0.0086
        GEE                RatingProduct  0.0178 0.0095 0.0616 -0.0009  0.0365
OLS-cluster                    Intercept -0.1180 0.0709 0.0960 -0.2570  0.0210
OLS-cluster          Valence[T.Negative]  0.1430 0.0698 0.0405  0.0062  0.2798
OLS-cluster                 Session[T.2] -0.0832 0.0485 0.0861 

In [13]:
# Cell 9b — Effect Size Calculation
print("=== Practical Effect Sizes (Session 1) ===")
res = fit_gee("FAA ~ C(Valence)*Momentum + RatingProduct", df_primary[df_primary.Session == 1])
b_mom = res.params["Momentum"]
b_int = res.params["C(Valence)[T.Negative]:Momentum"]
sd_faa = df_primary["FAA"].std()

pos_change = b_mom * 4
neg_change = (b_mom + b_int) * 4

print(f"Positive Shift (|Δ| 1->5): ΔFAA = {pos_change:+.4f} ({pos_change/sd_faa:+.3f} SD)")
print(f"Negative Shift (|Δ| 1->5): ΔFAA = {neg_change:+.4f} ({neg_change/sd_faa:+.3f} SD)")


=== Practical Effect Sizes (Session 1) ===
Positive Shift (|Δ| 1->5): ΔFAA = +0.2914 (+0.573 SD)
Negative Shift (|Δ| 1->5): ΔFAA = -0.0554 (-0.109 SD)


In [14]:
# Cell 10 — TABLE 2: Tonic session effects (FAA_pre / FAA_post / dFAA ~ Session)
for outcome in ["FAA_pre", "FAA", "dFAA", "PosteriorAlpha"]:
    _ = report(f"{outcome} ~ Session", df_f, f"TABLE 2 — {outcome} ~ Session")
    print()

TABLE 2 — FAA_pre ~ Session
  formula: FAA_pre ~ Session
--------------------------------------------------------------------------------------------
  Estimator         Term    Coef     SE      p   CI2.5  CI97.5
        GEE    Intercept  0.1599 0.0520 0.0021  0.0580  0.2619
        GEE Session[T.2] -0.0924 0.0499 0.0643 -0.1902  0.0055
OLS-cluster    Intercept  0.1614 0.0520 0.0019  0.0595  0.2634
OLS-cluster Session[T.2] -0.0969 0.0509 0.0569 -0.1966  0.0029

TABLE 2 — FAA ~ Session
  formula: FAA ~ Session
--------------------------------------------------------------------------------------------
  Estimator         Term    Coef     SE      p   CI2.5  CI97.5
        GEE    Intercept  0.1050 0.0324 0.0012  0.0415  0.1684
        GEE Session[T.2] -0.0888 0.0410 0.0303 -0.1692 -0.0085
OLS-cluster    Intercept  0.0996 0.0320 0.0018  0.0370  0.1623
OLS-cluster Session[T.2] -0.0906 0.0428 0.0341 -0.1745 -0.0068

TABLE 2 — dFAA ~ Session
  formula: dFAA ~ Session
-------------------------

In [15]:
# Cell 10b — Tonic independence: posterior alpha is distinct from FAA (Results, Tonic section)
# (1) within-participant correlation between FAA and posterior alpha (mean per-subject Pearson r)
wr = df_f.groupby("Subject_ID").apply(lambda d: d["FAA"].corr(d["PosteriorAlpha"]))
print(f"within-participant corr(FAA, PosteriorAlpha): mean r = {wr.mean():+.3f}  (n={int(wr.notna().sum())} subj)")
# (2) participants showing the Session 2 posterior-alpha increase
pa = df_f.groupby(["Subject_ID","Session"], observed=True)["PosteriorAlpha"].mean().unstack()
inc = int(((pa[2]-pa[1])>0).sum()); n = int(pa.dropna().shape[0])
print(f"participants with higher posterior alpha in Session 2: {inc} of {n}")
# (3) posterior-alpha session effect controlling for FAA (independence from frontal asymmetry)
r = fit_gee("PosteriorAlpha ~ Session + FAA", df_f)
print(f"PosteriorAlpha ~ Session + FAA:  Session2 b={r.params['Session[T.2]']:+.4f} p={r.pvalues['Session[T.2]']:.4f}")

within-participant corr(FAA, PosteriorAlpha): mean r = -0.043  (n=19 subj)
participants with higher posterior alpha in Session 2: 15 of 19
PosteriorAlpha ~ Session + FAA:  Session2 b=+0.1267 p=0.0017


In [16]:
# Cell 10c — Sensitivity: absolute-alpha FAA reproduces relative-alpha FAA (A4)
print("corr(relative FAA, absolute-alpha FAA) =", round(df_f["FAA"].corr(df_f["FAA_abs"]), 4))
d1 = df_f[df_f.Session == 1].copy()
d1["Valence"] = pd.Categorical(d1["Valence"].astype(str), categories=["Positive", "Negative"])
for dv in ["FAA", "FAA_abs"]:
    r = fit_gee(f"{dv} ~ C(Valence)*Momentum + RatingProduct", d1)
    t = "C(Valence)[T.Negative]:Momentum"
    print(f"  S1 {dv:8s} Val x Mom: b={r.params[t]:+.4f} p={r.pvalues[t]:.4f}")
for dv in ["FAA", "FAA_abs"]:
    r = fit_gee(f"{dv} ~ Session", df_f)
    print(f"  pooled {dv:8s} Session2: b={r.params['Session[T.2]']:+.4f} p={r.pvalues['Session[T.2]']:.4f}")


corr(relative FAA, absolute-alpha FAA) = 0.6253
  S1 FAA      Val x Mom: b=-0.0895 p=0.0120
  S1 FAA_abs  Val x Mom: b=-0.0338 p=0.1522
  pooled FAA      Session2: b=-0.0888 p=0.0303
  pooled FAA_abs  Session2: b=-0.0674 p=0.1097


In [17]:
# Cell 11 — TABLE 3: Session-stratified CDM
for s in (1, 2):
    sub = df_primary[df_primary.Session == s].copy()
    sub["Valence"] = pd.Categorical(sub["Valence"].astype(str), categories=["Positive","Negative"])
    _ = report("FAA ~ Valence*Momentum + RatingProduct", sub,
               f"TABLE 3 — Session {s} (N={len(sub)}, subj={sub.Subject_ID.nunique()})")
    print()

TABLE 3 — Session 1 (N=421, subj=19)
  formula: FAA ~ Valence*Momentum + RatingProduct
--------------------------------------------------------------------------------------------
  Estimator                         Term    Coef     SE      p   CI2.5  CI97.5
        GEE                    Intercept -0.1503 0.1031 0.1448 -0.3524  0.0517
        GEE          Valence[T.Negative]  0.2243 0.0851 0.0084  0.0575  0.3911
        GEE                     Momentum  0.0729 0.0380 0.0553 -0.0017  0.1474
        GEE Valence[T.Negative]:Momentum -0.0867 0.0335 0.0096 -0.1524 -0.0211
        GEE                RatingProduct  0.0180 0.0139 0.1956 -0.0093  0.0452
OLS-cluster                    Intercept -0.1743 0.1181 0.1398 -0.4057  0.0571
OLS-cluster          Valence[T.Negative]  0.2478 0.0904 0.0061  0.0707  0.4249
OLS-cluster                     Momentum  0.0924 0.0411 0.0245  0.0119  0.1729
OLS-cluster Valence[T.Negative]:Momentum -0.1019 0.0350 0.0036 -0.1704 -0.0334
OLS-cluster                Rat

  Estimator                         Term    Coef     SE      p   CI2.5  CI97.5
        GEE                    Intercept -0.1438 0.0964 0.1358 -0.3327  0.0451
        GEE          Valence[T.Negative]  0.0322 0.0858 0.7070 -0.1359  0.2003
        GEE                     Momentum  0.0283 0.0277 0.3065 -0.0259  0.0825
        GEE Valence[T.Negative]:Momentum -0.0127 0.0413 0.7593 -0.0936  0.0683
        GEE                RatingProduct  0.0202 0.0123 0.1006 -0.0039  0.0443
OLS-cluster                    Intercept -0.1594 0.0989 0.1072 -0.3533  0.0345
OLS-cluster          Valence[T.Negative]  0.0322 0.0882 0.7152 -0.1407  0.2050
OLS-cluster                     Momentum  0.0307 0.0284 0.2800 -0.0250  0.0863
OLS-cluster Valence[T.Negative]:Momentum -0.0100 0.0420 0.8127 -0.0923  0.0724
OLS-cluster                RatingProduct  0.0207 0.0125 0.0983 -0.0038  0.0453



In [18]:
# Cell 11b — Electrode specificity (B-S4): Session-1 Valence x Momentum by frontal pair
# F3/F4 is the pre-specified primary pair (FAA literature); F7/F8 and AF3/AF4 are exploratory.
d1 = df_primary[df_primary.Session == 1].copy()
d1["Valence"] = pd.Categorical(d1["Valence"].astype(str), categories=["Positive", "Negative"])
ITERM = "C(Valence)[T.Negative]:Momentum"
print("Session 1 Valence x Momentum interaction by frontal pair (GEE + RatingProduct):")
for name, col in [("F3/F4 (primary)", "FAA"), ("F7/F8", "FAA_F7F8"), ("AF3/AF4", "FAA_AF3AF4")]:
    r = fit_gee(f"{col} ~ C(Valence)*Momentum + RatingProduct", d1)
    print(f"  {name:16s} b={r.params[ITERM]:+.4f} SE={r.bse[ITERM]:.4f} p={r.pvalues[ITERM]:.4f}")


Session 1 Valence x Momentum interaction by frontal pair (GEE + RatingProduct):
  F3/F4 (primary)  b=-0.0867 SE=0.0335 p=0.0096
  F7/F8            b=+0.0232 SE=0.0550 p=0.6733
  AF3/AF4          b=-0.0658 SE=0.0580 p=0.2568


In [19]:
# Cell 12 — 3-way GEE habituation test (NOW PRINTED)
r3 = fit_gee("FAA ~ Valence*Momentum*Session + RatingProduct", df_primary)
print("=== 3-Way Interaction GEE: FAA ~ Valence*Momentum*Session + RatingProduct ===")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(tidy(r3, "GEE").to_string(index=False))
key = [t for t in r3.params.index if t.count(":") == 2]
if key:
    t = key[0]
    print(f"\n>> 3-way term [{t}]: beta={r3.params[t]:.4f}, SE={r3.bse[t]:.4f}, p={r3.pvalues[t]:.4f}")

=== 3-Way Interaction GEE: FAA ~ Valence*Momentum*Session + RatingProduct ===
Estimator                                      Term    Coef     SE      p   CI2.5  CI97.5
      GEE                                 Intercept -0.1724 0.0835 0.0389 -0.3360 -0.0088
      GEE                       Valence[T.Negative]  0.2427 0.0810 0.0027  0.0839  0.4016
      GEE                              Session[T.2]  0.0460 0.1071 0.6678 -0.1640  0.2560
      GEE          Valence[T.Negative]:Session[T.2] -0.2204 0.1011 0.0293 -0.4185 -0.0222
      GEE                                  Momentum  0.0860 0.0367 0.0192  0.0140  0.1580
      GEE              Valence[T.Negative]:Momentum -0.0994 0.0295 0.0008 -0.1572 -0.0416
      GEE                     Momentum:Session[T.2] -0.0635 0.0394 0.1071 -0.1407  0.0137
      GEE Valence[T.Negative]:Momentum:Session[T.2]  0.0969 0.0416 0.0198  0.0154  0.1783
      GEE                             RatingProduct  0.0181 0.0096 0.0608 -0.0008  0.0369

>> 3-way term [Valenc

In [20]:
# Cell 12b — Small-sample robustness (A3e): LMM (random intercept/participant) + participant permutation
# Complements GEE/OLS-cluster inference given only 19 participant clusters.
import statsmodels.formula.api as smf
CDM_S1 = "FAA ~ C(Valence)*Momentum + RatingProduct"
ITERM = "C(Valence)[T.Negative]:Momentum"

print("=== LMM (random intercept / participant) ===")
mp = smf.mixedlm("FAA ~ C(Valence)*Momentum + RatingProduct + C(Session)", df_primary,
                 groups=df_primary["Subject_ID"]).fit(reml=True)
print("Pooled CDM:")
for t in ["C(Session)[T.2]", "Momentum", "C(Valence)[T.Negative]", ITERM, "RatingProduct"]:
    print(f"  {t:34s} b={mp.params[t]:+.4f} SE={mp.bse[t]:.4f} p={mp.pvalues[t]:.4f}")
d1 = df_primary[df_primary.Session == 1].copy()
d1["Valence"] = pd.Categorical(d1["Valence"].astype(str), categories=["Positive", "Negative"])
m1 = smf.mixedlm(CDM_S1, d1, groups=d1["Subject_ID"]).fit(reml=True)
print("Session 1 CDM:")
for t in ["Momentum", "C(Valence)[T.Negative]", ITERM, "RatingProduct"]:
    print(f"  {t:34s} b={m1.params[t]:+.4f} SE={m1.bse[t]:.4f} p={m1.pvalues[t]:.4f}")

# Participant-level permutation of the Session-1 Valence x Momentum interaction
# (shuffle Valence within participant; cluster-robust OLS coefficient as the statistic)
obs = smf.ols(CDM_S1, data=d1).fit(cov_type="cluster", cov_kwds={"groups": d1["Subject_ID"]}).params[ITERM]
d1r = d1.reset_index(drop=True)
sub_pos = {s: np.where(d1r.Subject_ID.values == s)[0] for s in d1r.Subject_ID.unique()}
val = d1r["Valence"].astype(str).to_numpy()
N_PERM = 5000; rng = np.random.default_rng(0); null = np.empty(N_PERM)
for k in range(N_PERM):
    v = val.copy()
    for s, idx in sub_pos.items():
        v[idx] = rng.permutation(v[idx])
    dd = d1r.copy(); dd["Valence"] = pd.Categorical(v, categories=["Positive", "Negative"])
    null[k] = smf.ols(CDM_S1, data=dd).fit(cov_type="cluster",
                                           cov_kwds={"groups": dd["Subject_ID"]}).params[ITERM]
p_perm = (np.sum(np.abs(null) >= abs(obs)) + 1) / (N_PERM + 1)
print(f"\n=== Participant permutation (N={N_PERM}, within-subject Valence shuffle, seed=0) ===")
print(f"Observed S1 interaction (OLS-cluster) b={obs:+.4f}; two-sided empirical p={p_perm:.4f}")


=== LMM (random intercept / participant) ===
Pooled CDM:
  C(Session)[T.2]                    b=-0.0797 SE=0.0351 p=0.0233
  Momentum                           b=+0.0428 SE=0.0244 p=0.0801
  C(Valence)[T.Negative]             b=+0.1263 SE=0.0679 p=0.0626
  C(Valence)[T.Negative]:Momentum    b=-0.0450 SE=0.0303 p=0.1377
  RatingProduct                      b=+0.0178 SE=0.0101 p=0.0774


Session 1 CDM:
  Momentum                           b=+0.0643 SE=0.0351 p=0.0673
  C(Valence)[T.Negative]             b=+0.2137 SE=0.0931 p=0.0216
  C(Valence)[T.Negative]:Momentum    b=-0.0802 SE=0.0421 p=0.0568
  RatingProduct                      b=+0.0188 SE=0.0139 p=0.1770


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



=== Participant permutation (N=5000, within-subject Valence shuffle, seed=0) ===
Observed S1 interaction (OLS-cluster) b=-0.1019; two-sided empirical p=0.0068


In [21]:
# Cell 12c — Reference/band sensitivity (B-S1, C2): CSD and individualized-alpha FAA
# Transparency checks only; the main analyses use average-referenced, fixed 8-13 Hz FAA.
# CSD with a 15-electrode montage and a narrowed individual-alpha band are both under-powered (see Discussion/Limitations).
def _iaf_band(ep):
    e = ep.copy().crop(0.0, 3.0)
    roi = [e.ch_names.index(c) for c in ["P3", "Pz", "P4"]]
    psd = e.compute_psd(method="welch", fmin=5, fmax=15, n_fft=512, verbose=False)
    p = psd.get_data()[:, roi, :].mean((0, 1)); fr = psd.freqs; m = (fr >= 7) & (fr <= 13)
    pk = fr[m][np.argmax(p[m])]
    return max(7.0, pk - 2.0), min(13.0, pk + 2.0)

rec = []
for ep in price_epochs_list:
    if ep is None or len(ep) == 0:
        continue
    ep_csd = mne.preprocessing.compute_current_source_density(ep.copy(), verbose=False)
    ec = ep_csd.copy().crop(0.0, 3.0); f3, f4 = ec.ch_names.index("F3"), ec.ch_names.index("F4")
    ac = ec.compute_psd(method="welch", fmin=8, fmax=13, n_fft=512, verbose=False).get_data().mean(-1)
    faa_csd = np.log(np.abs(ac[:, f4])) - np.log(np.abs(ac[:, f3]))
    lo, hi = _iaf_band(ep); e2 = ep.copy().crop(0.0, 3.0)
    g3, g4 = e2.ch_names.index("F3"), e2.ch_names.index("F4")
    aa = e2.compute_psd(method="welch", fmin=lo, fmax=hi, n_fft=512, verbose=False).get_data().mean(-1)
    tt = e2.compute_psd(method="welch", fmin=1, fmax=40, n_fft=512, verbose=False).get_data().mean(-1)
    faa_iaf = np.log(aa[:, g4] / tt[:, g4]) - np.log(aa[:, g3] / tt[:, g3])
    md = ep.metadata.reset_index(drop=True)
    for j in range(len(ep)):
        m = md.iloc[j]
        rec.append(dict(Subject_ID=f"S{int(m['subject_id']):02d}", Session=int(m['session']),
                        Valence="Positive" if m['diff_score'] > 0 else "Negative",
                        Momentum=float(m['momentum']), RatingProduct=float(m['rating_product']),
                        FAA_csd=float(faa_csd[j]), FAA_iaf=float(faa_iaf[j])))
sens = pd.DataFrame(rec)
sens = sens[sens.Subject_ID.isin(keep)]  # same 19-participant screen as df_f
d1 = sens[sens.Session == 1].copy()
d1["Valence"] = pd.Categorical(d1["Valence"].astype(str), categories=["Positive", "Negative"])
t = "C(Valence)[T.Negative]:Momentum"
for dv in ["FAA_csd", "FAA_iaf"]:
    r = fit_gee(f"{dv} ~ C(Valence)*Momentum + RatingProduct", d1)
    print(f"S1 {dv:8s} Val x Mom: b={r.params[t]:+.4f} p={r.pvalues[t]:.4f}")
print("Primary average-referenced fixed-band FAA: p=0.012. CSD (15-ch) and narrowed IAF are under-powered (Limitations).")


S1 FAA_csd  Val x Mom: b=+0.0054 p=0.8409
S1 FAA_iaf  Val x Mom: b=-0.0322 p=0.4925
Primary average-referenced fixed-band FAA: p=0.012. CSD (15-ch) and narrowed IAF are under-powered (Limitations).


In [22]:
# Cell 12d — Further robustness of the Session-1 interaction (B-S2 boundary, B-S8 item, B-S12 ordinal, C3 leverage)
import statsmodels.formula.api as smf
ITERM = "C(Valence)[T.Negative]:Momentum"
d1 = df_primary[df_primary.Session == 1].copy()
d1["Valence"] = pd.Categorical(d1["Valence"].astype(str), categories=["Positive", "Negative"])
def ols_c(d, g):
    return smf.ols("FAA ~ C(Valence)*Momentum + RatingProduct", data=d).fit(cov_type="cluster", cov_kwds={"groups": d[g]})

# (B-S2) INCLUDE scale-boundary (as sensitivity since primary excludes) trials (9->9, 1->1)
bmask = ((df_f.RatingProduct == 9) & (df_f.RatingPrice == 9)) | ((df_f.RatingProduct == 1) & (df_f.RatingPrice == 1))
print(f"B-S2 boundary trials: {int(bmask.sum())} (9->9={int(((df_f.RatingProduct==9)&(df_f.RatingPrice==9)).sum())}, 1->1={int(((df_f.RatingProduct==1)&(df_f.RatingPrice==1)).sum())})")
dnb = df_f[df_f.Session == 1].copy() # N=937
dnb["Valence"] = pd.Categorical(dnb["Valence"].astype(str), categories=["Positive", "Negative"])
r = fit_gee("FAA ~ C(Valence)*Momentum + RatingProduct", dnb)
print(f"     including boundary (N={len(dnb)}): b={r.params[ITERM]:+.4f} p={r.pvalues[ITERM]:.4f}")

# (B-S8) item-level dependence: cluster by item + crossed RE (subject + item)
print(f"B-S8 unique items: {df_f.ImageID.nunique()} (S1={df_f[df_f.Session==1].ImageID.nunique()}, S2={df_f[df_f.Session==2].ImageID.nunique()})")
ri = ols_c(d1, "ImageID"); print(f"     cluster by ITEM: b={ri.params[ITERM]:+.4f} p={ri.pvalues[ITERM]:.4f}")
try:
    mf = smf.mixedlm("FAA ~ C(Valence)*Momentum + RatingProduct", d1, groups=np.ones(len(d1)),
                     vc_formula={"subj": "0+C(Subject_ID)", "item": "0+C(ImageID)"}).fit(reml=True, method="lbfgs")
    print(f"     crossed-RE (subj+item): b={mf.params[ITERM]:+.4f} p={mf.pvalues[ITERM]:.4f}")
except Exception as e:
    print("     crossed-RE failed:", e)

# (B-S12) ordinal / rank-based Momentum
d1r = d1.copy(); d1r["MomRank"] = d1r["Momentum"].rank()
rr = fit_gee("FAA ~ C(Valence)*MomRank + RatingProduct", d1r)
ti = "C(Valence)[T.Negative]:MomRank"
print(f"B-S12 rank(Momentum): b={rr.params[ti]:+.5f} p={rr.pvalues[ti]:.4f}")

# (C3) leave-one-participant-out + winsorize
ps = [ols_c(d1[d1.Subject_ID != s], "Subject_ID").pvalues[ITERM] for s in d1.Subject_ID.unique()]
print(f"C3 LOPO: {sum(p<0.05 for p in ps)}/{len(ps)} folds p<0.05 (max p={max(ps):.4f})")
d1w = d1.copy(); lo, hi = d1w.FAA.quantile([.01, .99]); d1w["FAA"] = d1w.FAA.clip(lo, hi)
rw = fit_gee("FAA ~ C(Valence)*Momentum + RatingProduct", d1w)
print(f"C3 winsorized(1/99%): b={rw.params[ITERM]:+.4f} p={rw.pvalues[ITERM]:.4f}")


B-S2 boundary trials: 117 (9->9=13, 1->1=104)
     including boundary (N=474): b=-0.0895 p=0.0120
B-S8 unique items: 60 (S1=30, S2=30)
     cluster by ITEM: b=-0.1019 p=0.0092
     crossed-RE (subj+item): b=-0.0753 p=0.0753
B-S12 rank(Momentum): b=-0.00091 p=0.0016
C3 LOPO: 19/19 folds p<0.05 (max p=0.0361)
C3 winsorized(1/99%): b=-0.0759 p=0.0122


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [23]:
# Cell 12e — Price-magnitude (B-S5) and participant-sex (B-S6) covariate checks (Session-1 interaction)
ITERM = "C(Valence)[T.Negative]:Momentum"
key = lambda s: str(s).replace(".jpg", "")

# (B-S5) within-category price-deviation covariate from price.csv
price = pd.read_csv(f"{DATA}/price.csv")
pm = dict(zip(price["image_id"].map(key), price["price"]))
tmp = df_primary.copy()
tmp["price"] = tmp["ImageID"].map(lambda s: pm.get(key(s), np.nan))
tmp["cat"] = tmp["ImageID"].map(lambda s: key(s).split("_")[-1])   # e.g. "10_electric" -> "electric"
tmp["logprice"] = np.log(tmp["price"])
tmp["PriceZ"] = tmp.groupby("cat")["logprice"].transform(lambda x: (x - x.mean()) / x.std())
print(f"[B-S5] price merged: {tmp.price.notna().sum()}/{len(tmp)} trials; categories: {sorted(tmp.cat.unique())}")
d1p = tmp[tmp.Session == 1].copy()
d1p["Valence"] = pd.Categorical(d1p["Valence"].astype(str), categories=["Positive", "Negative"])
r = fit_gee("FAA ~ C(Valence)*Momentum + RatingProduct + PriceZ", d1p)
print(f"[B-S5] S1 interaction +PriceZ: b={r.params[ITERM]:+.4f} p={r.pvalues[ITERM]:.4f} | "
      f"PriceZ p={r.pvalues['PriceZ']:.4f} | corr(PriceZ,Momentum)={d1p['PriceZ'].corr(d1p['Momentum']):.3f}")

# (B-S6) participant-sex covariate from sex.csv (de-identified subject_id [subjNN] -> SNN)
sexdf = pd.read_csv(f"{DATA}/sex.csv")  # columns: subject_id, sex
sexmap = {f"S{int(re.search(r'(\d+)', str(sid)).group(1)):02d}": str(sx).lower()
          for sid, sx in zip(sexdf["subject_id"], sexdf["sex"])}
# Report the sex composition of the ANALYZED sample (df_f, N=19), not the 21 recruited.
analyzed_sex = df_f.drop_duplicates("Subject_ID")["Subject_ID"].map(sexmap)
print(f"[B-S6] sex (analyzed N={df_f.Subject_ID.nunique()}): {analyzed_sex.value_counts().to_dict()}")
ts = df_primary.copy(); ts["Sex"] = ts["Subject_ID"].map(sexmap)
d1s = ts[ts.Session == 1].copy()
d1s["Valence"] = pd.Categorical(d1s["Valence"].astype(str), categories=["Positive", "Negative"])
d1s["Sex"] = pd.Categorical(d1s["Sex"], categories=["male", "female"])
r = fit_gee("FAA ~ C(Valence)*Momentum + RatingProduct + C(Sex)", d1s)
print(f"[B-S6] S1 interaction +Sex: b={r.params[ITERM]:+.4f} p={r.pvalues[ITERM]:.4f} | "
      f"Sex(female) b={r.params['C(Sex)[T.female]']:+.4f} p={r.pvalues['C(Sex)[T.female]']:.4f}")


[B-S5] price merged: 820/820 trials; categories: ['bag', 'electric', 'furniture']
[B-S5] S1 interaction +PriceZ: b=-0.0927 p=0.0049 | PriceZ p=0.1895 | corr(PriceZ,Momentum)=0.180
[B-S6] sex (analyzed N=19): {'male': 12, 'female': 7}


[B-S6] S1 interaction +Sex: b=-0.0892 p=0.0033 | Sex(female) b=+0.1781 p=0.0120


## Objective 3 — Single-trial ML classification across Momentum thresholds

In [24]:
# Cell 13 — Single-trial classification of Valence from FAA across Momentum thresholds
# Accuracy, balanced accuracy, ROC AUC, within-subject permutation p, class counts,
# and LOGO fold-level detail (#participants, single-class held-out folds). [A3f / B-S14]
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_auc_score, balanced_accuracy_score

thresholds = [1, 2, 3, 4, 5]
logo = LeaveOneGroupOut()

def run_logo(X, y, g):
    yt, yp, ypr, folds, one_class = [], [], [], 0, 0
    for tr, te in logo.split(X, y, g):
        if len(np.unique(y[tr])) < 2:
            continue
        if len(np.unique(y[te])) < 2:
            one_class += 1
        clf = LogisticRegression(class_weight="balanced", random_state=42).fit(X[tr], y[tr])
        yt += list(y[te]); yp += list(clf.predict(X[te])); ypr += list(clf.predict_proba(X[te])[:, 1])
        folds += 1
    return np.array(yt), np.array(yp), np.array(ypr), folds, one_class

rows = []
for th in thresholds:
    d = df_f[df_f.Momentum >= th]
    X = d[["FAA"]].values; y = (d.Valence == "Positive").astype(int).values; g = d.Subject_ID.values
    pos, neg, nsub = int(y.sum()), int((1 - y).sum()), d.Subject_ID.nunique()
    yt, yp, ypr, folds, one_class = run_logo(X, y, g)
    if len(np.unique(yt)) < 2:
        rows.append((th, len(d), pos, neg, np.nan, np.nan, np.nan, np.nan, nsub, one_class)); continue
    acc = accuracy_score(yt, yp); bal = balanced_accuracy_score(yt, yp); auc = roc_auc_score(yt, ypr)
    rng = np.random.default_rng(100 + th); null = []
    for _ in range(1000):
        yperm = y.copy()
        for s in np.unique(g):
            idx = np.where(g == s)[0]; yperm[idx] = rng.permutation(yperm[idx])
        if len(np.unique(yperm)) < 2: continue
        yt2, _, ypr2, _, _ = run_logo(X, yperm, g)
        if len(np.unique(yt2)) == 2: null.append(roc_auc_score(yt2, ypr2))
    null = np.array(null); perm_p = (np.sum(null >= auc) + 1) / (len(null) + 1)
    rows.append((th, len(d), pos, neg, acc, bal, auc, perm_p, nsub, one_class))

res = pd.DataFrame(rows, columns=["Thr","N","Pos","Neg","Acc","BalAcc","AUC","perm_p","subjects","single_class_folds"])
with pd.option_context("display.float_format", "{:.3f}".format):
    print(res.to_string(index=False))

# Participant bootstrap 95% CI at the primary threshold (|d|>=1)
d = df_f[df_f.Momentum >= 1]; subs = d.Subject_ID.unique(); rng = np.random.default_rng(7)
accs, bals, aucs = [], [], []
for _ in range(2000):
    parts = []
    for k, s in enumerate(rng.choice(subs, size=len(subs), replace=True)):
        t = d[d.Subject_ID == s].copy(); t["__g"] = f"{s}_{k}"; parts.append(t)
    db = pd.concat(parts, ignore_index=True)
    X = db[["FAA"]].values; y = (db.Valence == "Positive").astype(int).values; g = db["__g"].values
    yt, yp, ypr, _, _ = run_logo(X, y, g)
    if len(np.unique(yt)) == 2:
        accs.append(accuracy_score(yt, yp)); bals.append(balanced_accuracy_score(yt, yp)); aucs.append(roc_auc_score(yt, ypr))
pc = lambda a: (np.percentile(a, 2.5), np.percentile(a, 97.5))
print(f"\nParticipant bootstrap 95% CI (|d|>=1, 2000 resamples):")
print(f"  Accuracy [{pc(accs)[0]:.3f}, {pc(accs)[1]:.3f}]  BalAcc [{pc(bals)[0]:.3f}, {pc(bals)[1]:.3f}]  AUC [{pc(aucs)[0]:.3f}, {pc(aucs)[1]:.3f}]")


 Thr   N  Pos  Neg   Acc  BalAcc   AUC  perm_p  subjects  single_class_folds
   1 937  458  479 0.537   0.537 0.528   0.093        19                   0
   2 415  209  206 0.455   0.456 0.419   0.855        19                   0
   3 187   73  114 0.406   0.378 0.392   0.921        17                   4
   4 114   32   82 0.474   0.415 0.379   0.859        17                  11
   5  59   18   41 0.458   0.392 0.317   0.977        15                  11



Participant bootstrap 95% CI (|d|>=1, 2000 resamples):
  Accuracy [0.478, 0.556]  BalAcc [0.477, 0.556]  AUC [0.471, 0.558]


## Figures — Fig 3 (stratified hybrid CDM + tonic) & S1 Fig (pooled hybrid CDM)

Model lines = OLS cluster-robust; the interaction p on each panel is computed from GEE. Figures are written to `PLOS_ONE_V16_2/fig/`.

In [25]:
# Cell 14 — Generate Fig 3 (stratified hybrid CDM + tonic) and S1 Fig (pooled hybrid CDM)
# Model lines = OLS cluster-robust prediction (marginal over Session for the pooled panel);
# interaction p is COMPUTED from the GEE fit (reproduces Table 1 / Table 3).
# PLOS-ready output: RGB (no alpha), width <= 2250 px @ 300 dpi, TIFF (LZW) + PNG, fonts 8-12 pt.
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.cov_struct import Exchangeable
from patsy import build_design_matrices
from matplotlib.gridspec import GridSpec
from PIL import Image

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "figure.dpi": 100,
})
OUT_DIR = "PLOS_ONE_V16/fig"
os.makedirs(OUT_DIR, exist_ok=True)

def _save_rgb(fig, stem, dpi=300):
    """Save figure as PNG (preview) and TIFF (PLOS: RGB 8-bit, no alpha, LZW)."""
    png = f"{OUT_DIR}/{stem}.png"; tif = f"{OUT_DIR}/{stem}.tif"
    fig.savefig(png, dpi=dpi, facecolor="white")          # fixed canvas = figsize*dpi (no bbox tight)
    im = Image.open(png).convert("RGB")                   # drop alpha -> RGB
    im.save(png, dpi=(dpi, dpi))
    im.save(tif, format="TIFF", compression="tiff_lzw", dpi=(dpi, dpi))
    return im.size

def _standardize(df):
    df = df.copy()
    if "Session" in df.columns:
        df["Session"] = pd.Categorical(df["Session"].astype(str).str.strip().replace(
            {"Session1": "1", "S1": "1", "Session2": "2", "S2": "2"}), categories=["1", "2"])
    if "Valence" in df.columns:
        df["Valence"] = pd.Categorical(df["Valence"].astype(str).str.strip(), categories=["Positive", "Negative"])
    return df

def binned_means_cluster_bootstrap(df, bins_use, x="Momentum", y="FAA", subj="Subject_ID", n_boot=2000, seed=0):
    rng = np.random.default_rng(seed); bins = np.array(bins_use); mids = (bins[:-1] + bins[1:]) / 2.0
    out = []
    for dec in ["Positive", "Negative"]:
        d = df[df["Valence"] == dec].copy()
        if d.empty:
            continue
        d["_b"] = pd.cut(d[x], bins=bins_use, labels=False, include_lowest=True)
        for b in range(len(mids)):
            db = d[d["_b"] == b]; n = len(db)
            if n < 2:
                continue
            subs = db[subj].unique(); obs = db[y].mean(); bm = []
            for _ in range(n_boot):
                sb = rng.choice(subs, size=len(subs), replace=True); v = []
                for s in sb:
                    v.extend(db[db[subj] == s][y].values)
                if v:
                    bm.append(np.mean(v))
            if bm:
                lo, hi = np.percentile(bm, [2.5, 97.5])
                out.append(dict(Valence=dec, mid=mids[b], mean=obs, lo=lo, hi=hi, n=n))
    return out

def predicted_lines(df, formula, x="Momentum", rating="RatingProduct", grid=None):
    ols = smf.ols(formula, data=df).fit(cov_type="cluster", cov_kwds={"groups": df["Subject_ID"]})
    gee = GEE.from_formula(formula, groups="Subject_ID", data=df, cov_struct=Exchangeable()).fit()
    cand = [t for t in gee.pvalues.index if (":" in t and x in t and "Valence" in t)]
    p_int = gee.pvalues[cand[0]] if cand else np.nan
    if grid is None:
        grid = np.linspace(df[x].min(), df[x].max(), 50)
    mr = df[rating].mean(); sc = df["Session"].value_counts(); fr = sc / sc.sum()
    cov = ols.cov_params().values; pred = {d: {k: [] for k in "xylh"} for d in ("Positive", "Negative")}
    for dec in ("Positive", "Negative"):
        for xv in grid:
            Xb = None
            for w, sess in zip(fr.values, fr.index):
                row = pd.DataFrame({x: [xv], "Valence": [dec], rating: [mr], "Session": [sess]})
                row["Session"] = pd.Categorical(row["Session"], categories=df["Session"].cat.categories)
                row["Valence"] = pd.Categorical(row["Valence"], categories=df["Valence"].cat.categories)
                Xd = build_design_matrices([ols.model.data.design_info], row, return_type="dataframe")[0]
                Xb = Xd.values * w if Xb is None else Xb + Xd.values * w
            yh = float((Xb @ ols.params.values.reshape(-1, 1))[0, 0]); se = float(np.sqrt(Xb @ cov @ Xb.T)[0, 0])
            pred[dec]["x"].append(xv); pred[dec]["y"].append(yh)
            pred[dec]["l"].append(yh - 1.96 * se); pred[dec]["h"].append(yh + 1.96 * se)
    for dec in pred:
        for k in pred[dec]:
            pred[dec][k] = np.array(pred[dec][k])
    return pred, p_int

def plot_hybrid(ax, df, title, formula, bins=None, n_boot=2000, seed=0):
    df = _standardize(df); CP, CN = "#1f77b4", "#d62728"
    pred, p_int = predicted_lines(df, formula)
    ax.plot(pred["Positive"]["x"], pred["Positive"]["y"], lw=2.4, color=CP, label="Model: Positive Shift", zorder=3)
    ax.fill_between(pred["Positive"]["x"], pred["Positive"]["l"], pred["Positive"]["h"], color=CP, alpha=0.08, zorder=1)
    ax.plot(pred["Negative"]["x"], pred["Negative"]["y"], lw=2.4, ls="--", color=CN, label="Model: Negative Shift", zorder=3)
    ax.fill_between(pred["Negative"]["x"], pred["Negative"]["l"], pred["Negative"]["h"], color=CN, alpha=0.08, zorder=1)
    yref = list(pred["Positive"]["y"]) + list(pred["Negative"]["y"])
    if bins is not None:
        for d in binned_means_cluster_bootstrap(df, bins, n_boot=n_boot, seed=seed):
            yref.append(d["mean"]); c = CP if d["Valence"] == "Positive" else CN
            fmt = "o" if d["Valence"] == "Positive" else "s"
            off = -0.14 if d["Valence"] == "Positive" else 0.14
            ax.errorbar(d["mid"] + off, d["mean"], yerr=[[d["mean"] - d["lo"]], [d["hi"] - d["mean"]]],
                        fmt=fmt, color=c, ms=6, capsize=3, elinewidth=1.2, alpha=0.9, zorder=4)
            side = -1 if d["Valence"] == "Positive" else 1   # n-label to outer side of marker (clears the vertical CI bar; avoids overlap)
            ax.annotate(f"n={d['n']}", (d["mid"] + off, d["mean"]), textcoords="offset points",
                        xytext=(9 * side, 0), color=c, ha=("right" if side < 0 else "left"), va="center",
                        fontsize=7, fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.75), zorder=6)
    ax.set_title(title, fontweight="bold", pad=8)
    ax.set_xlabel(r"Decision Momentum ($|\Delta|$)", labelpad=5)
    ax.set_ylabel("Post-price FAA (0-3 s)", labelpad=5)
    ax.axhline(0, color="gray", lw=1.0, ls=":", zorder=1)
    ax.text(0.96, 0.04, f"Interaction $p={p_int:.4f}$", transform=ax.transAxes, fontsize=8,
            ha="right", va="bottom", bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))
    ax.legend(loc="upper left", framealpha=0.9, ncol=1)
    ax.grid(True, alpha=0.25, ls="--")
    ylo, yhi = float(np.min(yref)), float(np.max(yref)); pad = 0.16 + 0.12 * (yhi - ylo)
    return p_int, ylo - pad, yhi + pad

def plot_paired(ax, dft, ycol, title):
    df = dft.copy(); df["Session"] = df["Session"].astype(str).str.strip()
    for s in df["Subject_ID"].unique():
        ss = df[df["Subject_ID"] == s].sort_values("Session")
        if len(ss) == 2:
            ax.plot([1, 2], [ss[ss.Session == "1"][ycol].values[0], ss[ss.Session == "2"][ycol].values[0]],
                    marker="o", color="0.7", alpha=0.5, ms=4, lw=1.0)
    m1, m2 = df[df.Session == "1"][ycol].mean(), df[df.Session == "2"][ycol].mean()
    ax.plot([1, 2], [m1, m2], marker="D", lw=3.0, ms=7, color="k", zorder=5, label="Group mean")
    ax.set_title(title, fontweight="bold"); ax.set_xticks([1, 2]); ax.set_xticklabels(["Session 1", "Session 2"])
    ax.set_ylabel(ycol); ax.axhline(0, lw=0.8, alpha=0.4, color="k"); ax.legend(loc="upper right"); ax.grid(True, alpha=0.25, ls="--")

# ---- S1 Fig: pooled hybrid CDM (interaction p = pooled Valence x Momentum GEE; cf. Table 1) ----
df_plot_mom = _standardize(df_primary); df_plot_all = _standardize(df_f); bins_use = [0.5, 2.5, 4.5, 8.5]
fig1, ax1 = plt.subplots(figsize=(7.0, 5.0))
p_pool, lo, hi = plot_hybrid(ax1, df_plot_mom, "Pooled Hybrid CDM",
                             "FAA ~ C(Valence)*Momentum + RatingProduct + C(Session)", bins=bins_use)
ax1.set_ylim(lo, hi); plt.tight_layout()
sz = _save_rgb(fig1, "supplement_pool_hybrid_CDM"); plt.show(); plt.close(fig1)
print(f"Saved S1 Fig {sz}px  (pooled interaction p={p_pool:.4f})")

# ---- Fig 3: A/B stratified hybrid CDM (computed p; cf. Table 3) + C/D tonic paired ----
df_tonic = (df_plot_all.groupby(["Subject_ID", "Session"])[["FAA_pre", "FAA"]].mean()
            .reset_index().rename(columns={"FAA": "FAA_post"}))
df_s1, df_s2 = df_plot_mom[df_plot_mom.Session == "1"], df_plot_mom[df_plot_mom.Session == "2"]
sf = "FAA ~ C(Valence)*Momentum + RatingProduct"
fig3 = plt.figure(figsize=(7.3, 5.8))
gs = GridSpec(2, 2, figure=fig3, height_ratios=[1.1, 1.0], hspace=0.48, wspace=0.30)
axA, axB = fig3.add_subplot(gs[0, 0]), fig3.add_subplot(gs[0, 1])
axC, axD = fig3.add_subplot(gs[1, 0]), fig3.add_subplot(gs[1, 1])
pA, loA, hiA = plot_hybrid(axA, df_s1, "Session 1 (Hybrid CDM)", sf, bins=bins_use)
pB, loB, hiB = plot_hybrid(axB, df_s2, "Session 2 (Hybrid CDM)", sf, bins=bins_use)
slo, shi = min(loA, loB), max(hiA, hiB)            # shared y-axis for A & B (comparable slopes)
axA.set_ylim(slo, shi); axB.set_ylim(slo, shi)
plot_paired(axC, df_tonic, "FAA_pre", r"Baseline FAA$_{pre}$ (-1-0 s)")
plot_paired(axD, df_tonic, "FAA_post", r"Post-price FAA$_{post}$ (0-3 s)")
for ax, L in zip([axA, axB, axC, axD], "ABCD"):
    ax.text(-0.15, 1.05, L, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top")
sz3 = _save_rgb(fig3, "figure3"); plt.show(); plt.close(fig3)
print(f"Saved Fig 3  {sz3}px  (S1 interaction p={pA:.4f}, S2 interaction p={pB:.4f}; shared y=[{slo:.2f},{shi:.2f}])")


Saved S1 Fig (2100, 1500)px  (pooled interaction p=0.0932)


Saved Fig 3  (2190, 1740)px  (S1 interaction p=0.0096, S2 interaction p=0.7593; shared y=[-0.24,0.88])
